In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from sklearn.model_selection import train_test_split

# ── CONFIG ────────────────────────────────────────────────────────────────────

REAL_DIR    = '/kaggle/input/notebooks/francisbawa/753m-deepfake-auditor-image-pipeline/face_dataset/real'  # ← update this
FAKE_DIR    = '/kaggle/input/notebooks/francisbawa/753m-deepfake-auditor-image-pipeline/face_dataset/fake'  # ← update this
SAVE_PATH   = '/kaggle/working/deepfake_efficientnet_v4.pth'
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 20
LR          = 3e-4
WEIGHT_DECAY= 1e-4
PATIENCE    = 5
MIXUP_ALPHA = 0.2
UNFREEZE_EPOCH = 3
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── DATASET ───────────────────────────────────────────────────────────────────
class DeepfakeDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


# ── TRANSFORMS ────────────────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1)
    ], p=0.5),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
    ], p=0.3),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# ── DATA SPLIT ────────────────────────────────────────────────────────────────
real_paths = glob.glob(os.path.join(REAL_DIR, '*.jpg'))
fake_paths = glob.glob(os.path.join(FAKE_DIR, '*.jpg'))
all_paths  = real_paths + fake_paths
all_labels = [0] * len(real_paths) + [1] * len(fake_paths)

print(f"Total samples → Real: {len(real_paths)}, Fake: {len(fake_paths)}")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.2,
    stratify=all_labels,
    random_state=42
)

train_dataset = DeepfakeDataset(train_paths, train_labels, transform=train_transforms)
val_dataset   = DeepfakeDataset(val_paths,   val_labels,   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


# ── MODEL ─────────────────────────────────────────────────────────────────────
model = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

# Phase 1: freeze the backbone, only train the classifier head
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(DEVICE)


# ── LOSS / OPTIMIZER / SCHEDULER ──────────────────────────────────────────────
# No class weights needed — dataset is balanced 50/50
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


# ── MIXUP HELPERS ─────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ── TRAINING LOOP ─────────────────────────────────────────────────────────────
best_val_loss     = float('inf')
patience_counter  = 0
frozen            = True

print("\nStarting training...\n")

for epoch in range(1, EPOCHS + 1):

    # ── Phase 2: unfreeze full backbone after UNFREEZE_EPOCH ──────────────────
    if frozen and epoch >= UNFREEZE_EPOCH:
        print(f"  -- Epoch {epoch}: Unfreezing full backbone --")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=LR / 5, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - epoch + 1)
        frozen = False

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        images, labels_a, labels_b, lam = mixup_data(images, labels)
        optimizer.zero_grad()
        outputs = model(images)
        loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss  += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels_a.size(0)
        train_correct += (lam * (predicted == labels_a).float()
                          + (1 - lam) * (predicted == labels_b).float()).sum().item()

    scheduler.step()

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total   += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    # ── Metrics ───────────────────────────────────────────────────────────────
    t_loss = train_loss / len(train_loader)
    t_acc  = 100 * train_correct / train_total
    v_loss = val_loss / len(val_loader)
    v_acc  = 100 * val_correct / val_total

    all_preds   = np.array(all_preds)
    all_targets = np.array(all_targets)
    tp = ((all_preds == 1) & (all_targets == 1)).sum()
    fp = ((all_preds == 1) & (all_targets == 0)).sum()
    fn = ((all_preds == 0) & (all_targets == 1)).sum()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    print(f"Epoch [{epoch:02d}/{EPOCHS}] "
          f"| Train Loss: {t_loss:.4f} Acc: {t_acc:.2f}% "
          f"| Val Loss: {v_loss:.4f} Acc: {v_acc:.2f}% "
          f"| F1: {f1:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
            'val_acc': v_acc,
        }, SAVE_PATH)
        print(f"  New best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("\nEarly stopping triggered.")
            break

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
print(f"Model saved to: {SAVE_PATH}")

Using device: cuda:0
Total samples → Real: 12505, Fake: 12508
Train: 20010 | Val: 5003
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 189MB/s]



Starting training...

Epoch [01/20] | Train Loss: 0.6101 Acc: 71.11% | Val Loss: 0.5329 Acc: 80.09% | F1: 0.8112 | LR: 0.000298
  New best model saved (val_loss=0.5329)
Epoch [02/20] | Train Loss: 0.5691 Acc: 74.69% | Val Loss: 0.5090 Acc: 81.23% | F1: 0.8265 | LR: 0.000293
  New best model saved (val_loss=0.5090)
  -- Epoch 3: Unfreezing full backbone --
Epoch [03/20] | Train Loss: 0.4993 Acc: 80.67% | Val Loss: 0.3745 Acc: 90.57% | F1: 0.9091 | LR: 0.000060
  New best model saved (val_loss=0.3745)
Epoch [04/20] | Train Loss: 0.4494 Acc: 84.77% | Val Loss: 0.3286 Acc: 92.74% | F1: 0.9282 | LR: 0.000058
  New best model saved (val_loss=0.3286)
Epoch [05/20] | Train Loss: 0.4220 Acc: 86.94% | Val Loss: 0.3043 Acc: 94.52% | F1: 0.9465 | LR: 0.000056
  New best model saved (val_loss=0.3043)
Epoch [06/20] | Train Loss: 0.4058 Acc: 88.22% | Val Loss: 0.2899 Acc: 95.50% | F1: 0.9558 | LR: 0.000053
  New best model saved (val_loss=0.2899)
Epoch [07/20] | Train Loss: 0.3822 Acc: 89.71% | Val 